### 拉取官方镜像（以 9.2.1 为例）
docker pull docker.elastic.co/elasticsearch/elasticsearch:9.2.1
### 启动容器（单节点，关闭安全认证便于测试）
docker run -d --name es-knn \
 -p 9200:9200 -p 9300:9300 \
 -e discovery.type=single-node \
 -e xpack.security.enabled=false \
  docker.elastic.co/elasticsearch/elasticsearch:9.2.1
### 安装Elasticsearch客户端
pip install elasticsearch  （ notebook内核用的是 D:\Code\LLM\.venv（uv 管理的虚拟环境），而 pip install elasticsearch 装到了系统 Python 3.11 里，所以内核里import不到）
或：cd D:\Code\LLM; uv add elasticsearch sentence-transformers

In [ ]:
# 导入依赖
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from sentence_transformers import SentenceTransformer

In [ ]:
# 1. 准备数据
items = [
    {
        "type": "phone",
        "id": "用户A",
        "content": "用户A的手机号是13800001234"
    },
    {
        "type": "phone",
        "id": "用户B",
        "content": "用户B的手机号是13800005678"
    },
    {
        "type": "order",
        "id": "订单1001",
        "content": "订单1001编号203011010001"
    },
    {
        "type": "order",
        "id": "订单1002",
        "content": "订单1002编号203011010123"
    }
]

In [ ]:
# 2. 加载 Embedding 模型 （拿到文本 -> 生成向量 -> 向量维度）
model = SentenceTransformer(r"D:\models\bge-small-zh-v1.5")

texts = [
    item["content"] for item in items
]

embeddings = model.encode(
    texts, 
    normalize_embeddings=True
)

dimension = len(embeddings[0])

print("向量维度：", dimension)

In [ ]:
# 3. 连接 Elasticsearch
es = Elasticsearch(
    "http://localhost:9200",
    request_timeout=30
)
# 探活
print("ES连接状态:",es.ping())

In [ ]:
# 4. 创建 Index
index_name = "knowledge_demo"

if es.indices.exists(index=index_name):         # =Miilvus的client.has_collection(name)
    es.indices.delete(index=index_name)

# 定义映射（ES 的表结构）
mappings = {
    "properties":{

        # 文本向量
        "embeddings": {
            "type": "dense_vector",      # 向量类型
            "dims": dimension,           # 维度
            "index": True,               # 建索引（可被搜索）
            "similarity": "cosine"       # 用余弦相似度,等价Milvus的 metric_type="COSINE"
        },

        # 普通字段
        "type": {"type": "keyword"},
        "item_id": {"type": "keyword"},
        "content": {"type": "text"}
    }
}

es.indices.create(index=index_name, mappings=mappings)

In [ ]:
# 5. 准备向量数据
actions = []

for item, vector in zip(items, embeddings):
    actions.append(
        {
            "_index": index_name,
            "_id": item["id"],
            "_source": {
                "type": item["type"],
                "item_id": item["id"],
                "content": item["content"],
                "embedding": vector.tolist()
            }
        }
    )

In [ ]:
# 6. 批量写入 ES
bulk(es, actions)

es.indices.refresh(index=index_name)

print(f"已写入 {len(items)} 条数据")

In [ ]:
# 7. 查询
query_text = "用户A的电话号码是多少?"

query_vector = model.encode(
    [query_text],
    normalize_embeddings=True
)[0].tolist()                       # 取出第一行转换为list

k = 3                               # 定义 Top-K （返回结果数）

In [ ]:
# 8. 混合搜索
result = es.search(
    index=index_name,

    # ① 关键词/过滤器搜索
    query={
        "bool": {
            "should": [
                {"match": {"content": query_text}}   # 关键词匹配正文
            ],
            "filter": [
                {"term": {"type": "phone"}}          # 只搜 phone 类
            ]
        }
    },

    # ② KNN 向量搜索
    knn={
        "field": "embedding",                  # 在哪个向量字段上搜
        "query_vector": query_vector,          # 查询向量
        "k": k,                                # 返回 Top-K
        "num_candidates": 100                  # 候选池大小
        "boost": 0.5                           # 向量分数权重
    },
    source=["type",  "item_id", "content"]     # 返回哪些字段
)

In [ ]:
# 9. 输出结果
print(f"\n查询:{query_text}")
print(f"Top-{k} 相似结果：")

for rank, hit in enumerate(results["hits"]["hits"], start=1):   # 第一层 hits:"搜索命中"大板块,第二层 hits:拿到里面的结果数组
    source = hit["_source"]
    print(
        f"{rank}. "
        f"score={hit['_score']:.4f} | "
        f"type={source['type']} | "
        f"id={source['item_id']} | "
        f"content={source['content']}"
    )